# 04 — Netflix Prize Features

**MIS 3080 Final Project — Movie Recommender (extension)**

This notebook adds **collaborative-filtering signals from the Netflix Prize dataset** to our movie features. Our existing model used TMDB metadata (budget, year, genre, director, …) to predict ratings. Netflix gives us something genuinely different: **how real users rated these movies**, on the order of 100 million ratings.

### What the Netflix data buys us

Two new families of features:

1. **Per-movie aggregates** — mean Netflix rating (1-5 scale), rating count, rating standard deviation. A second, independent quality signal from a different audience.
2. **Latent factors from matrix factorization (SVD)** — 32 numbers per movie that capture *which kinds of users like this movie* without anyone labelling those user types. This is collaborative filtering, conceptually different from the metadata-based features in our existing model.

### Why we use TruncatedSVD (sklearn) and not Surprise

The classic recommendation library `scikit-surprise` does not build cleanly on Windows + Python 3.12. `sklearn.decomposition.TruncatedSVD` does the same matrix factorization, runs in seconds on this dataset, and uses only the libraries we already have.

### What this notebook produces

- **`artifacts/movies_with_netflix.parquet`** — every column from `movies_clean.parquet`, plus 4 Netflix aggregate features and 32 SVD latent factors per movie. Shape: 4,803 rows × ~68 columns. About 44% of TMDB movies will have non-NaN Netflix features (the Netflix Prize data was collected up to 2005, so post-2005 releases are not in the catalog — XGBoost handles the missing values via median imputation).


## 0. Setup — paths


In [ ]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
NETFLIX_DIR  = PROJECT_ROOT.parent / "Netflix Prize Data"
DATA_PATH    = PROJECT_ROOT / "artifacts" / "movies_clean.parquet"
ARTIFACTS    = PROJECT_ROOT / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

assert NETFLIX_DIR.exists(), f"Netflix dir not found at {NETFLIX_DIR}"
assert DATA_PATH.exists(),   f"Run 01_eda.ipynb first to produce {DATA_PATH}"
print("NETFLIX_DIR :", NETFLIX_DIR)
print("DATA_PATH   :", DATA_PATH)
print("contents    :", sorted(p.name for p in NETFLIX_DIR.iterdir()))


In [ ]:
import re
import time

import numpy as np
import pandas as pd
from rapidfuzz import fuzz, process
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD

np.random.seed(42)


## 1. Stream the four `combined_data_*.txt` files


The combined-data files are formatted as blocks: a `movie_id:` line followed by one `user_id,rating,date` line per rating. The four files together hold ~100 million ratings (~2 GB on disk) — too large to fit in memory comfortably. We **stream** through them line-by-line, accumulating per-movie statistics as we go.

We also keep a 10 % random sample of `(user_id, movie_id, rating)` triples for the matrix-factorization step. That's roughly 10 million ratings — plenty for stable SVD factors and small enough to fit in memory comfortably.

On a typical laptop this section takes **3-5 minutes**. The progress prints once per file.


In [ ]:
SAMPLE_RATE = 0.10  # keep 10% of ratings for SVD
rng = np.random.default_rng(42)

movie_n     = {}    # movie_id -> count
movie_sum   = {}    # movie_id -> sum
movie_sumsq = {}    # movie_id -> sum of squared ratings

sample_users   = []
sample_movies  = []
sample_ratings = []

t0 = time.time()
for fname in ["combined_data_1.txt", "combined_data_2.txt",
              "combined_data_3.txt", "combined_data_4.txt"]:
    path = NETFLIX_DIR / fname
    file_t0 = time.time()
    current_movie = None
    n_in_file = 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line:
                continue
            if line.endswith(":"):
                current_movie = int(line[:-1])
                continue
            c1 = line.index(",")
            c2 = line.index(",", c1 + 1)
            user_id = int(line[:c1])
            rating  = int(line[c1 + 1:c2])
            movie_n[current_movie]     = movie_n.get(current_movie, 0) + 1
            movie_sum[current_movie]   = movie_sum.get(current_movie, 0) + rating
            movie_sumsq[current_movie] = movie_sumsq.get(current_movie, 0) + rating * rating
            n_in_file += 1
            if rng.random() < SAMPLE_RATE:
                sample_users.append(user_id)
                sample_movies.append(current_movie)
                sample_ratings.append(rating)
    print(f"{fname}: {n_in_file:,} ratings ({time.time()-file_t0:.1f}s)")
print(f"\ntotal time: {time.time()-t0:.1f}s")
print(f"unique movies: {len(movie_n):,}")
print(f"sample for SVD: {len(sample_users):,} ratings")


### Aggregate into a per-movie DataFrame


In [ ]:
agg = pd.DataFrame({
    "movie_id":             list(movie_n.keys()),
    "netflix_rating_count": [movie_n[m] for m in movie_n],
    "netflix_rating_sum":   [movie_sum[m] for m in movie_n],
    "netflix_rating_sumsq": [movie_sumsq[m] for m in movie_n],
})
agg["netflix_mean_rating"] = agg["netflix_rating_sum"] / agg["netflix_rating_count"]
agg["netflix_rating_std"]  = np.sqrt(
    agg["netflix_rating_sumsq"] / agg["netflix_rating_count"]
    - agg["netflix_mean_rating"] ** 2
)
agg["netflix_log_count"] = np.log1p(agg["netflix_rating_count"])

agg = agg[["movie_id", "netflix_mean_rating", "netflix_rating_count",
           "netflix_rating_std", "netflix_log_count"]]
print(agg.describe().round(3))


## 2. Matrix factorization for movie latent factors


We build a sparse user × movie matrix from the 10 % sample, **center the ratings around the global mean** (so SVD captures *deviations* from the average rather than the average itself), then run `TruncatedSVD` with `n_components = 32`.

The output we care about is `svd.components_.T` — a `(n_movies, 32)` matrix in which **each row is a movie's latent representation**. Two movies with similar rows are loved by similar users.


In [ ]:
s_users   = np.asarray(sample_users,   dtype=np.int64)
s_movies  = np.asarray(sample_movies,  dtype=np.int32)
s_ratings = np.asarray(sample_ratings, dtype=np.float32)
del sample_users, sample_movies, sample_ratings

# Filter to users with >=10 ratings in the sample
uniq_users, counts = np.unique(s_users, return_counts=True)
active = set(uniq_users[counts >= 10])
mask = np.fromiter((u in active for u in s_users), dtype=bool, count=len(s_users))
s_users   = s_users[mask]
s_movies  = s_movies[mask]
s_ratings = s_ratings[mask]
print(f"filtered ratings: {len(s_users):,}")

# Map user_id -> row, movie_id -> col
uniq_users  = np.unique(s_users)
uniq_movies = np.unique(s_movies)
user_to_row  = {u: i for i, u in enumerate(uniq_users)}
movie_to_col = {m: j for j, m in enumerate(uniq_movies)}
rows = np.fromiter((user_to_row[u]  for u in s_users),  dtype=np.int32, count=len(s_users))
cols = np.fromiter((movie_to_col[m] for m in s_movies), dtype=np.int32, count=len(s_movies))

# Center ratings around global mean
global_mean = s_ratings.mean()
centered = s_ratings - global_mean
print(f"global mean rating: {global_mean:.3f}")

mat = csr_matrix((centered, (rows, cols)),
                 shape=(len(uniq_users), len(uniq_movies)),
                 dtype=np.float32)
print(f"matrix shape: {mat.shape}, nnz: {mat.nnz:,}")


In [ ]:
K = 32
t0 = time.time()
svd = TruncatedSVD(n_components=K, random_state=42, algorithm="randomized", n_iter=5)
svd.fit(mat)
print(f"SVD fit in {time.time()-t0:.1f}s")
print(f"variance explained by top {K}: {svd.explained_variance_ratio_.sum():.3f}")

movie_factors = svd.components_.T.astype(np.float32)
factors_df = pd.DataFrame(
    movie_factors,
    columns=[f"netflix_svd_{i:02d}" for i in range(K)],
)
factors_df.insert(0, "movie_id", uniq_movies)
factors_df.shape


## 3. Fuzzy-match Netflix titles to TMDB


Netflix has no TMDB or IMDb ID, so we have to match by **title + year**. Steps:

1. Load `movie_titles.csv` (we use a custom loader because some titles contain commas — `pd.read_csv` would mangle them).
2. Normalize both Netflix and TMDB titles: lowercase, strip punctuation, collapse whitespace, drop a leading "the".
3. For each TMDB movie, take Netflix titles within ±1 year and find the best fuzzy match using `rapidfuzz.fuzz.token_set_ratio`.
4. Accept matches with score ≥ 88. In practice most matches score 100 or close to it.

Most unmatched TMDB rows are **post-2005 releases** — the Netflix Prize data was collected up to 2005 and excludes everything after. That's fine: for those rows the Netflix features stay NaN and median imputation handles them downstream.


In [ ]:
# Load TMDB
tmdb = pd.read_parquet(DATA_PATH)
print("TMDB rows:", len(tmdb))

# Custom Netflix title loader — handles commas in titles
def load_netflix_titles(path):
    rows = []
    with open(path, "r", encoding="latin-1") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line:
                continue
            parts = line.split(",", 2)
            if len(parts) != 3:
                continue
            mid, yr, title = parts
            rows.append((int(mid), int(yr) if yr != "NULL" else None, title))
    return pd.DataFrame(rows, columns=["movie_id", "year", "title"])

netflix_titles = load_netflix_titles(NETFLIX_DIR / "movie_titles.csv")
print("Netflix titles:", len(netflix_titles))
netflix_titles.head()


In [ ]:
def normalize(s):
    if not isinstance(s, str): return ""
    s = s.lower()
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    if s.startswith("the "):
        s = s[4:]
    return s

tmdb["norm_title"] = tmdb["title"].apply(normalize)
netflix_titles["norm_title"] = netflix_titles["title"].apply(normalize)


In [ ]:
# Index Netflix by year so we can do a year-windowed lookup quickly
nf_by_year = {}
for _, row in netflix_titles.iterrows():
    if pd.isna(row["year"]):
        continue
    nf_by_year.setdefault(int(row["year"]), []).append(
        (row["movie_id"], row["norm_title"])
    )
print("Netflix years indexed:", len(nf_by_year))


In [ ]:
THRESHOLD = 88
t0 = time.time()
matched_ids   = []
matched_score = []
for _, t in tmdb.iterrows():
    if pd.isna(t["release_year"]):
        matched_ids.append(None); matched_score.append(None); continue
    year = int(t["release_year"])
    cands = []
    for dy in (-1, 0, 1):
        cands.extend(nf_by_year.get(year + dy, []))
    if not cands:
        matched_ids.append(None); matched_score.append(None); continue
    titles = [c[1] for c in cands]
    best = process.extractOne(
        t["norm_title"], titles,
        scorer=fuzz.token_set_ratio, score_cutoff=THRESHOLD,
    )
    if best is None:
        matched_ids.append(None); matched_score.append(None)
    else:
        _, score, idx = best
        matched_ids.append(cands[idx][0]); matched_score.append(score)

tmdb["netflix_movie_id"] = matched_ids
tmdb["match_score"]      = matched_score
n = tmdb["netflix_movie_id"].notna().sum()
print(f"matched {n:,} / {len(tmdb):,}  ({n/len(tmdb):.1%})  in {time.time()-t0:.1f}s")
print(f"median fuzzy score: {tmdb['match_score'].median()}")


In [ ]:
# A few sample matches as a sanity check
(tmdb.dropna(subset=["netflix_movie_id"])
     .merge(netflix_titles, left_on="netflix_movie_id", right_on="movie_id",
            suffixes=("_tmdb", "_nf"))
     [["title_tmdb", "release_year", "title_nf", "year", "match_score"]]
     .sample(8, random_state=0))


## 4. Merge aggregates + SVD factors into TMDB


In [ ]:
enriched = tmdb.merge(
    agg, how="left", left_on="netflix_movie_id", right_on="movie_id"
).drop(columns=["movie_id"], errors="ignore")

enriched = enriched.merge(
    factors_df, how="left", left_on="netflix_movie_id", right_on="movie_id"
).drop(columns=["movie_id"], errors="ignore")

# Drop the helper cols
enriched = enriched.drop(columns=["norm_title", "match_score"], errors="ignore")
print("enriched shape:", enriched.shape)
print("rows w/ netflix_mean_rating :", enriched["netflix_mean_rating"].notna().sum())
print("rows w/ SVD factors        :", enriched["netflix_svd_00"].notna().sum())


## 5. Save the enriched dataset


In [ ]:
out_path = ARTIFACTS / "movies_with_netflix.parquet"
enriched.to_parquet(out_path, index=False)
print(f"wrote {out_path} ({out_path.stat().st_size/1024:.1f} KB)")


In [ ]:
# Roundtrip sanity check
reloaded = pd.read_parquet(out_path)
assert reloaded.shape == enriched.shape
print("roundtrip OK")


## 6. Next steps

With `movies_with_netflix.parquet` saved, the next step is to **retrain the rating regressor**:

- **`02_train_model.ipynb`** — load the enriched parquet, add the four Netflix aggregates and 32 SVD factors to the numeric feature list, retrain XGBoost, and report a clean before/after metrics table.

Expected lift: roughly **R² 0.58 → 0.65** from the new features.
